# Linear regression

Recover the line $y = 3x + 2$ from noisy points using `nn.Linear`, a loss function, an optimizer, and a training loop written out step by step.

In [ ]:
import matplotlib.pyplot as plt
import torch
from torch import nn

from pytorch_examples.data import make_linear_data
from pytorch_examples.utils import set_seed

set_seed(0)
x, y = make_linear_data(n=200, weight=3.0, bias=2.0, noise=0.1, seed=0)
print("x:", tuple(x.shape), " y:", tuple(y.shape))

In [ ]:
plt.figure(figsize=(5, 3))
plt.scatter(x, y, s=8)
plt.xlabel("x")
plt.ylabel("y")
plt.title("Noisy data")
plt.tight_layout()
plt.show()

## The model, loss and optimizer

`nn.Linear(1, 1)` holds one weight and one bias, both starting at random values. Mean squared error measures how far predictions are from the targets, and SGD nudges the parameters to reduce it.

In [ ]:
model = nn.Linear(1, 1)
loss_fn = nn.MSELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)

print("before training: weight=%.3f bias=%.3f" % (model.weight.item(), model.bias.item()))

## The training loop

Every iteration does the same five things: clear old gradients, predict, compute the loss, back-propagate, and update.

In [ ]:
losses = []
for epoch in range(200):
    optimizer.zero_grad()
    prediction = model(x)
    loss = loss_fn(prediction, y)
    loss.backward()
    optimizer.step()
    losses.append(loss.item())

print("after training:  weight=%.3f bias=%.3f" % (model.weight.item(), model.bias.item()))
print("final loss: %.5f" % losses[-1])

In [ ]:
grid = torch.linspace(-1, 1, 50).unsqueeze(1)
with torch.no_grad():
    fitted = model(grid)

fig, axes = plt.subplots(1, 2, figsize=(9, 3))
axes[0].plot(losses)
axes[0].set_title("Training loss")
axes[0].set_xlabel("epoch")
axes[0].set_yscale("log")
axes[1].scatter(x, y, s=8, label="data")
axes[1].plot(grid, fitted, color="red", label="fitted line")
axes[1].set_title("Fit")
axes[1].legend()
plt.tight_layout()
plt.show()

The learned parameters should land close to the true values of 3 and 2.

In [ ]:
assert abs(model.weight.item() - 3.0) < 0.2
assert abs(model.bias.item() - 2.0) < 0.2
print("Recovered the line.")